In [1]:
import torch
import numpy as np
from torch.utils.data import DataLoader
import torch.optim as optim
import torch.nn as nn
from torch_optimizer import Lookahead

#make root folder visible
import sys
sys.path.append('../../')

# from src.models.model import KeywordSpottingModel_with_cls
from src.data.data_loader import load_speech_commands_dataset, load_bg_noise_dataset
from src.utils.utils import set_memory_GB, print_model_size, log_to_file, plot_learning_curves, EarlyStopping
from src.utils.augmentations import add_time_shift_and_align, add_silence
from src.utils.train_utils import trainig_loop




In [2]:
from torch.utils.data import DataLoader, Dataset
import random
import numpy as np
import torch
from src.data.data_loader import mfcc, delta

class TFDatasetAdapter(Dataset):
    def __init__(self, tf_dataset, bg_noise_dataset, fixed_length, n_mfcc, n_fft, hop_length, n_mels,
                 augmentation=False, derivative=True, noise_level=0.0, MFCC_transform=True):
        self.tf_dataset = tf_dataset
        self.data = list(tf_dataset)
        self.bg_noise_data = list(bg_noise_dataset) if bg_noise_dataset is not None else None
        self.fixed_length = fixed_length
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.augmentation = augmentation
        self.derivative = derivative
        self.noise_level = noise_level
        self.MFCC_transform = MFCC_transform

        # Compute input_dim based on feature extraction settings
        if self.MFCC_transform:
            # If derivative is True, we stack the original MFCC with first and second order deltas.
            self.input_dim = self.n_mfcc * 3 if self.derivative else self.n_mfcc
        else:
            # For raw audio, we add a channel dimension to the waveform.
            self.input_dim = 1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio, label = self.data[idx]
        audio = audio.numpy()

        # Normalize the audio tensor
        audio = audio / np.max(np.abs(audio))
        audio = audio.astype(np.float32)

        # Squeeze extra dimensions if present
        if audio.ndim > 1:
            audio = np.squeeze(audio)

        # Add background noise if available
        if self.bg_noise_data:
            bg_noise_audio = random.choice(self.bg_noise_data)
            if len(bg_noise_audio) < len(audio):
                bg_noise_audio = np.pad(bg_noise_audio, (0, len(audio) - len(bg_noise_audio)), mode='constant')
            else:
                start_idx = random.randint(0, len(bg_noise_audio) - len(audio))
                bg_noise_audio = bg_noise_audio[start_idx:start_idx + len(audio)]
            audio = audio + self.noise_level * bg_noise_audio

        # Pad or trim the audio to the fixed length
        if len(audio) < self.fixed_length:
            audio = np.pad(audio, (0, self.fixed_length - len(audio)), mode='constant')
        else:
            audio = audio[:self.fixed_length]

        # If augmentation functions are provided, apply them
        if self.augmentation:
            for aug in self.augmentation:
                audio = aug(audio)

        if self.MFCC_transform:
            # Compute MFCC features
            audio = audio.astype(np.float32)
            MFCC = mfcc(y=audio, sr=16000, n_mfcc=self.n_mfcc, n_fft=self.n_fft,
                        hop_length=self.hop_length, n_mels=self.n_mels)
            if self.derivative:
                # Compute first and second order derivatives and stack them
                MFCC_delta = delta(MFCC)
                MFCC_delta2 = delta(MFCC, order=2)
                MFCC = np.vstack([MFCC, MFCC_delta, MFCC_delta2])
            # Remove extra dimensions if necessary
            output = MFCC
        else:
            # For raw waveform, add a channel dimension: shape becomes [1, fixed_length]
            output = np.expand_dims(audio, axis=0)

        return torch.tensor(output, dtype=torch.float32), torch.tensor(label.numpy(), dtype=torch.long)

In [3]:
import torch
import torch.nn as nn
from mamba_ssm import Mamba

class KeywordSpottingModel_with_cls(nn.Module):
    def __init__(self, input_dim, d_model, d_state, d_conv, expand, label_names,
                 num_mamba_layers=1, dropout_rate=0.2, use_cls_token=True):
        super(KeywordSpottingModel_with_cls, self).__init__()
        
        # Projection layer: input_dim comes from the dataset
        self.proj = nn.Linear(input_dim, d_model)  
        self.use_cls_token = use_cls_token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        
        # Build a stack of Mamba layers with normalization
        self.mamba_layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, expand=expand, d_conv=d_conv))
            self.layer_norms.append(nn.RMSNorm(d_model, eps=1e-5))

        self.fc = nn.Linear(d_model, len(label_names))
        self.dropout = nn.Dropout(dropout_rate)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        # x shape: [batch_size, input_dim, sequence_length]
        # Permute to [batch_size, sequence_length, input_dim] for projection
        x = x.permute(0, 2, 1)
        x = self.proj(x)

        if self.use_cls_token:
            batch_size = x.size(0)
            cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [batch_size, 1, d_model]
            x = torch.cat((x, cls_tokens), dim=1)
        
        # Permute back to [batch_size, d_model, sequence_length] for Mamba layers
        x = x.permute(0, 2, 1)
        for mamba_layer, layer_norm in zip(self.mamba_layers, self.layer_norms):
            x = mamba_layer(x)
            x = layer_norm(x)
        x = self.dropout(x)
        
        if self.use_cls_token:
            features = x[:, :, -1]
        else:
            features = self.pool(x).squeeze(-1)
        
        x = self.fc(features)
        return x

In [4]:
torch.cuda.is_available()

In [5]:
train_ds, val_ds, test_ds, silence_ds , info = load_speech_commands_dataset(reduced=True)

In [6]:
bg_noise_ds = None

In [7]:
label_names = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
print(label_names)

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torch_optimizer import Lookahead
import random
import numpy as np
import sys

# Make root folder visible
sys.path.append('../../')

# Import your modules and functions
from src.data.data_loader import load_speech_commands_dataset, load_bg_noise_dataset, mfcc, delta
from src.utils.augmentations import add_time_shift_and_align, add_silence
from src.utils.train_utils import trainig_loop

# Define the dataset adapter (as provided)
from torch.utils.data import Dataset

class TFDatasetAdapter(Dataset):
    def __init__(self, tf_dataset, bg_noise_dataset, fixed_length, n_mfcc, n_fft, hop_length, n_mels,
                 augmentation=False, derivative=True, noise_level=0.0, MFCC_transform=True):
        self.tf_dataset = tf_dataset
        self.data = list(tf_dataset)
        self.bg_noise_data = list(bg_noise_dataset) if bg_noise_dataset is not None else None
        self.fixed_length = fixed_length
        self.n_mfcc = n_mfcc
        self.n_fft = n_fft
        self.hop_length = hop_length
        self.n_mels = n_mels
        self.augmentation = augmentation
        self.derivative = derivative
        self.noise_level = noise_level
        self.MFCC_transform = MFCC_transform

        # Compute input_dim based on feature extraction settings.
        # If using MFCC with derivative, input_dim is n_mfcc * 3, else just n_mfcc.
        if self.MFCC_transform:
            self.input_dim = self.n_mfcc * 3 if self.derivative else self.n_mfcc
        else:
            # For raw audio, we add a channel dimension.
            self.input_dim = 1

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        audio, label = self.data[idx]
        audio = audio.numpy()

        # Normalize the audio tensor
        audio = audio / np.max(np.abs(audio))
        audio = audio.astype(np.float32)

        # Squeeze extra dimensions if present
        if audio.ndim > 1:
            audio = np.squeeze(audio)

        # Add background noise if available
        if self.bg_noise_data:
            bg_noise_audio = random.choice(self.bg_noise_data)
            if len(bg_noise_audio) < len(audio):
                bg_noise_audio = np.pad(bg_noise_audio, (0, len(audio) - len(bg_noise_audio)), mode='constant')
            else:
                start_idx = random.randint(0, len(bg_noise_audio) - len(audio))
                bg_noise_audio = bg_noise_audio[start_idx:start_idx + len(audio)]
            audio = audio + self.noise_level * bg_noise_audio

        # Pad or trim the audio to the fixed length
        if len(audio) < self.fixed_length:
            audio = np.pad(audio, (0, self.fixed_length - len(audio)), mode='constant')
        else:
            audio = audio[:self.fixed_length]

        # Apply augmentations if provided
        if self.augmentation:
            for aug in self.augmentation:
                audio = aug(audio)

        if self.MFCC_transform:
            # Compute MFCC features
            audio = audio.astype(np.float32)
            MFCC = mfcc(y=audio, sr=16000, n_mfcc=self.n_mfcc, n_fft=self.n_fft,
                        hop_length=self.hop_length, n_mels=self.n_mels)
            if self.derivative:
                # Compute first and second order derivatives and stack them
                MFCC_delta = delta(MFCC)
                MFCC_delta2 = delta(MFCC, order=2)
                MFCC = np.vstack([MFCC, MFCC_delta, MFCC_delta2])
            output = MFCC
        else:
            # For raw waveform, add a channel dimension.
            output = np.expand_dims(audio, axis=0)

        return torch.tensor(output, dtype=torch.float32), torch.tensor(label.numpy(), dtype=torch.long)

# Define your model (as provided)
from mamba_ssm import Mamba

class KeywordSpottingModel_with_cls(nn.Module):
    def __init__(self, input_dim, d_model, d_state, d_conv, expand, label_names,
                 num_mamba_layers=1, dropout_rate=0.2, use_cls_token=True):
        super(KeywordSpottingModel_with_cls, self).__init__()
        
        # Projection layer: input_dim comes from the dataset
        self.proj = nn.Linear(input_dim, d_model)  
        self.use_cls_token = use_cls_token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        
        # Build a stack of Mamba layers with normalization
        self.mamba_layers = nn.ModuleList()
        self.layer_norms = nn.ModuleList()
        for _ in range(num_mamba_layers):
            self.mamba_layers.append(Mamba(d_model=d_model, d_state=d_state, expand=expand, d_conv=d_conv))
            self.layer_norms.append(nn.RMSNorm(d_model, eps=1e-5))

        self.fc = nn.Linear(d_model, len(label_names))
        self.dropout = nn.Dropout(dropout_rate)
        self.pool = nn.AdaptiveAvgPool1d(1)

    def forward(self, x):
        # x shape: [batch_size, input_dim, sequence_length]
        # Permute to [batch_size, sequence_length, input_dim] for projection
        x = x.permute(0, 2, 1)
        x = self.proj(x)

        if self.use_cls_token:
            batch_size = x.size(0)
            cls_tokens = self.cls_token.expand(batch_size, -1, -1)  # [batch_size, 1, d_model]
            x = torch.cat((x, cls_tokens), dim=1)
        
        # Permute back to [batch_size, d_model, sequence_length] for Mamba layers
        x = x.permute(0, 2, 1)
        for mamba_layer, layer_norm in zip(self.mamba_layers, self.layer_norms):
            x = mamba_layer(x)
            x = layer_norm(x)
        x = self.dropout(x)
        
        if self.use_cls_token:
            features = x[:, :, -1]
        else:
            features = self.pool(x).squeeze(-1)
        
        x = self.fc(features)
        return x

# Load datasets
train_ds, val_ds, test_ds, silence_ds, info = load_speech_commands_dataset(reduced=True)
bg_noise_ds = None  # or load_bg_noise_dataset() if available
label_names = ['down', 'go', 'left', 'no', 'off', 'on', 'right', 'stop', 'up', 'yes']
print("Label Names:", label_names)

def run_ablation_tests():
    # Fixed dataset parameters
    fixed_length = 16000
    n_mfcc = 40
    n_fft = 512
    hop_length = 160
    n_mels = 40

    # Define different ablation settings.
    # Each experiment is a dictionary with settings for the dataset and model.
    experiments = [
        {
            "name": "with_derivative_and_cls",
            "derivative": True,
            "use_cls_token": True,
        },
        {
            "name": "no_derivative_and_with_cls",
            "derivative": False,
            "use_cls_token": True,
        },
        {
            "name": "with_derivative_and_no_cls",
            "derivative": True,
            "use_cls_token": False,
        },
        {
            "name": "no_derivative_and_no_cls",
            "derivative": False,
            "use_cls_token": False,
        }
    ]
    
    results = {}
    device = "cuda" if torch.cuda.is_available() else "cpu"
    
    for exp in experiments:
        print(f"\nRunning experiment: {exp['name']}")
        # Create dataset adapters with the current ablation setting for derivatives
        train_adapter = TFDatasetAdapter(
            tf_dataset=train_ds,
            bg_noise_dataset=bg_noise_ds,
            fixed_length=fixed_length,
            n_mfcc=n_mfcc,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            augmentation=[add_time_shift_and_align, add_silence],
            derivative=exp["derivative"],
            noise_level=0.1,
            MFCC_transform=True
        )
        val_adapter = TFDatasetAdapter(
            tf_dataset=val_ds,
            bg_noise_dataset=bg_noise_ds,
            fixed_length=fixed_length,
            n_mfcc=n_mfcc,
            n_fft=n_fft,
            hop_length=hop_length,
            n_mels=n_mels,
            augmentation=[add_time_shift_and_align, add_silence],
            derivative=exp["derivative"],
            noise_level=0.1,
            MFCC_transform=True
        )
        
        # Create DataLoaders
        train_loader = DataLoader(train_adapter, batch_size=32, shuffle=True)
        val_loader = DataLoader(val_adapter, batch_size=32, shuffle=False)
        
        # Compute d_model based on fixed_length and hop_length.
        # d_model = (fixed_length // hop_length) + (2 if using CLS token else 1)
        d_model_value = (fixed_length // hop_length) + (2 if exp["use_cls_token"] else 1)
        
        # Instantiate the model with the computed d_model and corresponding CLS token setting.
        input_dim = train_adapter.input_dim  # Already computed as n_mfcc * 3 if derivative, else n_mfcc
        model = KeywordSpottingModel_with_cls(
            input_dim=input_dim,
            d_model=d_model_value,
            d_state=64,
            d_conv=32,
            expand=4,
            label_names=label_names,
            num_mamba_layers=2,
            dropout_rate=0.2,
            use_cls_token=exp["use_cls_token"]
        )
        model.to(device)
        
        # Define optimizer and loss criterion
        optimizer = Lookahead(torch.optim.Adam(model.parameters(), lr=0.001))
        criterion = nn.CrossEntropyLoss()
        
        # Train the model using your training loop (assumed to return best validation accuracy)
        best_val_acc = trainig_loop(model, train_loader, val_loader, criterion, optimizer,
                                    num_epochs=20, device=device)
        results[exp["name"]] = best_val_acc
        print(f"Experiment {exp['name']} best validation accuracy: {best_val_acc}")
    
    print("\nAblation Test Results:")
    for exp_name, acc in results.items():
        print(f"{exp_name}: {acc}")

if __name__ == "__main__":
    run_ablation_tests()